# GPS + LGBM OD-Flow Experiments

Takes pre-trained GPS models (from `gps_od.ipynb`) and trains a LightGBM decoder
on top of the frozen GPS encoder embeddings via `train_lgbm_from_model`.

**Workflow:**
1. Run `gps_od.ipynb` to train and save GPS weights
2. Run this notebook to extract embeddings and train LGBM head
3. Compare GPS-only vs GPS+LGBM results


In [ ]:
import sys, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.insert(0, str(Path('.').resolve()))

from models.GPS.config import (
    device, cleanup_gpu,
    WEIGHTS_DIR, WEIGHTS_CPC_BEST_DIR, SINGLE_CITY_IDS,
    load_model_config,
)
from models.GPS.data_load import prepare_single_city_data
from models.GPS.model import make_model
from models.GPS.lgbm_pipeline import train_lgbm_from_model
from models.GPS.metrics import predict_full_matrix
from models.shared.metrics import canonical_od_metrics
from benchmarking.repeats import single_city_run_id

print(f'Device: {device}')
print(f'Weights dir: {WEIGHTS_DIR}')


## Configuration

Select which GPS run_ids to use as encoder donors for LGBM training.
Run `gps_od.ipynb` first to generate weights for the desired configs.


In [ ]:
# Base GPS run_ids whose encoder will be used as LGBM feature extractor.
# These must match runs produced by gps_od.ipynb.
DONOR_WEIGHTS_DIR = WEIGHTS_DIR  # switch to WEIGHTS_CPC_BEST_DIR for CPC-best GPS checkpoints
LGBM_CITY_ID = SINGLE_CITY_IDS[0]
LGBM_DONOR_BASE_IDS = [
    'SC_BL_CE_RLE_LAPE',
    'SC_BL_CE_NORM_LAPE',
    'SC_BL_HUBER_LOG_NORM_LAPE',
    'SC_BL_MULTITASK_LOG_NORM_LAPE',
]
LGBM_DONOR_IDS = [single_city_run_id(base_id, LGBM_CITY_ID) for base_id in LGBM_DONOR_BASE_IDS]

trained = [d for d in LGBM_DONOR_IDS if (DONOR_WEIGHTS_DIR / f'{d}.pt').exists()]
print(f'City: {LGBM_CITY_ID}')
print(f'Donor weights dir: {DONOR_WEIGHTS_DIR}')
print(f'Donor run_ids: {len(LGBM_DONOR_IDS)}')
print(f'Available (weights found): {len(trained)}')
if len(trained) < len(LGBM_DONOR_IDS):
    missing = [d for d in LGBM_DONOR_IDS if d not in trained]
    print(f'Missing (run gps_od.ipynb first): {missing}')


## Data Loading

Load city data (cached per city + pe_type + pair_split_mode — same logic as gps_od.ipynb).


In [ ]:
_data_cache = {}

def get_city_data(area_id=LGBM_CITY_ID, pe_type='rwpe', pair_split_mode='nonzero_pairs'):
    key = (area_id, pe_type, pair_split_mode)
    if key not in _data_cache:
        print(f'  Loading single-city data (city={area_id}, pe_type={pe_type}, split={pair_split_mode})...')
        _data_cache[key] = prepare_single_city_data(
            area_id=area_id, pe_type=pe_type, pair_split_mode=pair_split_mode)
        N = _data_cache[key]['num_nodes']
        train_fit_pairs = _data_cache[key]['train_fit_mask'].sum()
        train_nz_pairs = _data_cache[key]['train_mask'].sum()
        print(f'  N={N}, train_fit_pairs={train_fit_pairs}, train_nz_pairs={train_nz_pairs}')
    return _data_cache[key]


## LGBM Training Loop

For each donor GPS model: load weights → extract embeddings → train LGBM.


In [ ]:
lgbm_results = {}      # lgbm_run_id -> result dict from train_lgbm_from_model
gps_results_cache = {} # donor_id -> GPS metrics (loaded alongside for comparison)

for donor_id in LGBM_DONOR_IDS:
    weight_path = DONOR_WEIGHTS_DIR / f'{donor_id}.pt'
    saved_cfg   = load_model_config(donor_id, weights_dir=DONOR_WEIGHTS_DIR)

    if not weight_path.exists():
        print(f'  [SKIP] {donor_id}: weights not found — run gps_od.ipynb first')
        continue
    if saved_cfg is None:
        print(f'  [SKIP] {donor_id}: no saved config JSON')
        continue

    city_data = get_city_data(LGBM_CITY_ID, saved_cfg.pe_type, saved_cfg.pair_split_mode)

    # Load GPS model
    gps_model = make_model(saved_cfg, graph_data_ref=city_data['graph_data'])
    gps_model.load_state_dict(torch.load(str(weight_path), map_location=device))
    gps_model.to(device).eval()

    # GPS-only metrics (for comparison)
    with torch.no_grad():
        pred_gps = predict_full_matrix(gps_model, city_data, saved_cfg)
    pred_gps[pred_gps < 0] = 0
    gps_results_cache[donor_id] = canonical_od_metrics(
        pred_gps,
        city_data['od_matrix_np'],
        test_mask=city_data.get('test_mask'),
        test_full_mask=city_data.get('test_full_mask'),
        train_mask=city_data.get('train_mask'),
        val_mask=city_data.get('val_mask'),
        train_full_mask=city_data.get('train_full_mask'),
        val_full_mask=city_data.get('val_full_mask'),
    )
    print(f'  GPS   {donor_id}: CPC_full={gps_results_cache[donor_id]["CPC_full"]:.4f}')

    # Train LGBM on GPS embeddings
    lgbm_run_id = f'{donor_id}_lgbm'
    try:
        result = train_lgbm_from_model(lgbm_run_id, city_data, gps_model, donor_id)
        lgbm_results[lgbm_run_id] = result
    except Exception as e:
        print(f'  ERROR {donor_id}: {e}')
    finally:
        del gps_model; cleanup_gpu()

print(f'\nCompleted: {len(lgbm_results)} LGBM models')


In [ ]:
# Models are saved automatically by train_lgbm_from_model.
# Show which .lgbm files are now in WEIGHTS_DIR:
saved = sorted(WEIGHTS_DIR.glob('*_lgbm.lgbm'))
print(f'Saved LGBM models ({len(saved)}):')
for p in saved:
    print(f'  {p.name}')


## Results Comparison

GPS-encoder-only vs GPS-encoder + LGBM-decoder.


In [ ]:
rows = {}

for lgbm_id, r in lgbm_results.items():
    donor_id = lgbm_id.replace('_lgbm', '')
    gps_m    = gps_results_cache.get(donor_id, {})
    lgbm_m   = r.get('metrics_canonical', {})

    rows[f'{donor_id}'] = {
        'model':    'GPS (neural)',
        'CPC_full':  gps_m.get('CPC_full',  float('nan')),
        'MAE_full':  gps_m.get('MAE_full',  float('nan')),
        'RMSE_full': gps_m.get('RMSE_full', float('nan')),
    }
    rows[f'{donor_id}_lgbm'] = {
        'model':    'GPS emb + LGBM',
        'CPC_full':  lgbm_m.get('CPC_full',  float('nan')),
        'MAE_full':  lgbm_m.get('MAE_full',  float('nan')),
        'RMSE_full': lgbm_m.get('RMSE_full', float('nan')),
    }

if rows:
    df = pd.DataFrame(rows).T
    df = df.sort_values('CPC_full', ascending=False)
    print('\n' + '=' * 80)
    print(f'  GPS vs GPS+LGBM (city={LGBM_CITY_ID}, sorted by CPC_full)')
    print('=' * 80)
    print(df.to_string())

    import os; os.makedirs('results', exist_ok=True)
    out_path = f'results/lgbm_comparison__{LGBM_CITY_ID}.csv'
    df.to_csv(out_path)
    print(f'\nSaved to {out_path}')
else:
    print('No results to display.')


## Training Curves

LGBM iteration loss curves.


In [ ]:
# LightGBM doesn't have iteration curves by default; show bar chart instead
if rows:
    df_plot = pd.DataFrame(rows).T
    gps_rows  = df_plot[df_plot['model'] == 'GPS (neural)']
    lgbm_rows = df_plot[df_plot['model'] == 'GPS emb + LGBM']

    x = range(len(gps_rows))
    labels = [idx.replace(f'__city_{LGBM_CITY_ID}', '').replace('SC_', '') for idx in gps_rows.index]
    w = 0.35

    fig, ax = plt.subplots(figsize=(max(8, len(x)*1.5), 5))
    ax.bar([i - w/2 for i in x], gps_rows['CPC_full'].astype(float),
           width=w, label='GPS (neural)', color='#2196F3')
    ax.bar([i + w/2 for i in x], lgbm_rows['CPC_full'].astype(float),
           width=w, label='GPS emb + LGBM', color='#FF9800')
    ax.set_xticks(list(x)); ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylabel('CPC_full'); ax.set_title(f'GPS vs GPS+LGBM - CPC_full comparison ({LGBM_CITY_ID})')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()
